# PINN advantage #2: Parametric surrogate (one network, infinitely many solutions)

Solve linear convection $u_t + c\,u_x = 0$ **for a whole family of speeds** $c$ at once,
by making $c$ an **extra input** to the network: $N(x, t, c)$.

**Why a PINN wins here:** a classical solver produces one solution per run — change $c$,
rerun the whole simulation. The parametric PINN learns the map $(x,t,c)\mapsto u$ once;
afterwards, evaluating the solution for *any* $c$ is a single, millisecond forward pass.
This is exactly what a **surrogate model / digital twin** needs for real-time design,
optimization, or uncertainty quantification over the parameter.

Ground truth (used only to score, never to train): $u(x,t)=u_0(x-c\,t)$ with $u_0$ a
Gaussian. Runs in a few seconds on Colab.

In [ ]:
# Cell 1 -- Imports, device, problem + parameter range
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

L, T = 2.0, 1.0
C_MIN, C_MAX = 0.5, 2.0        # <-- the network will cover this whole range of speeds

def u0(x):                     # smooth Gaussian initial profile
    exp = torch.exp if torch.is_tensor(x) else np.exp
    return exp(-((x - 0.5) ** 2) / (2 * 0.12 ** 2))

def exact(x, t, c):            # analytic solution for a given c (scoring only)
    return u0(x - c * t)

### Key idea — the parameter is just another coordinate
The network takes **three** inputs $(x, t, c)$ instead of two. During training we sample
$c$ randomly from $[c_\min, c_\max]$ *alongside* $x$ and $t$, and — crucially — the PDE
residual uses **that same sampled $c$**:
$$\text{residual} = u_t + c\,u_x .$$
So every collocation point teaches the network the physics *for its own value of $c$*.
The initial condition $u(x,0)=u_0(x)$ holds for all $c$, which ties the family together.

In [ ]:
# Cell 2 -- Network with (x, t, c) inputs
class ParametricPINN(nn.Module):
    def __init__(self, h=48):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, t, c], dim=1))

model = ParametricPINN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
mse = nn.MSELoss()

def rand(n, lo, hi, grad=False):
    x = torch.rand(n, 1, device=device) * (hi - lo) + lo
    return x.requires_grad_(True) if grad else x

In [ ]:
# Cell 3 -- Train over the joint (x, t, c) domain
EPOCHS = 5000
t0 = time.perf_counter()
for e in range(EPOCHS):
    opt.zero_grad()

    # --- PDE residual on random (x, t, c) ---
    xi = rand(3000, 0, L, grad=True)
    ti = rand(3000, 0, T, grad=True)
    ci = rand(3000, C_MIN, C_MAX)               # sampled speed (an input, not differentiated)
    ui = model(xi, ti, ci)
    u_t = torch.autograd.grad(ui, ti, torch.ones_like(ui), create_graph=True)[0]
    u_x = torch.autograd.grad(ui, xi, torch.ones_like(ui), create_graph=True)[0]
    loss_pde = mse(u_t + ci * u_x, torch.zeros_like(ui))

    # --- Initial condition u(x,0)=u0(x), for every c ---
    x_ic = rand(1500, 0, L); c_ic = rand(1500, C_MIN, C_MAX)
    loss_ic = mse(model(x_ic, torch.zeros_like(x_ic), c_ic), u0(x_ic))

    # --- Inflow boundary at x=0 (Gaussian has essentially left; g approx 0), for every c ---
    t_bc = rand(1500, 0, T); c_bc = rand(1500, C_MIN, C_MAX)
    x_bc = torch.zeros_like(t_bc)
    loss_bc = mse(model(x_bc, t_bc, c_bc), exact(x_bc, t_bc, c_bc))

    loss = loss_pde + 10.0 * loss_ic + 10.0 * loss_bc
    loss.backward(); opt.step()
    if e % 500 == 0:
        print(f'epoch {e:4d}  pde {loss_pde.item():.2e}  ic {loss_ic.item():.2e}  bc {loss_bc.item():.2e}')
if device.type == 'cuda':
    torch.cuda.synchronize()
train_time = time.perf_counter() - t0
print(f'\nOne-time training: {train_time:.2f} s (covers ALL c in [{C_MIN}, {C_MAX}])')

### The payoff — query many $c$ values instantly, no re-solving
A classical CFD code would rerun the full time-stepping for each new $c$. The trained
surrogate answers any $c$ in one forward pass. Below we pick several $c$ (including
values the network was never explicitly asked about) and time a single query.

In [ ]:
# Cell 4 -- Evaluate the surrogate at several c and compare to exact
x = np.linspace(0, L, 200)
xe = torch.tensor(x, dtype=torch.float32, device=device).reshape(-1, 1)
c_list = [0.6, 1.0, 1.4, 1.8]

# time a single query
with torch.no_grad():
    ce = torch.full_like(xe, 1.0)
    if device.type == 'cuda': torch.cuda.synchronize()
    tq = time.perf_counter()
    _ = model(xe, torch.full_like(xe, T), ce)
    if device.type == 'cuda': torch.cuda.synchronize()
    query_time = time.perf_counter() - tq
print(f'Single-c query time: {query_time*1e3:.2f} ms  (vs a full CFD re-run per c)')

plt.figure(figsize=(11, 5))
for c in c_list:
    with torch.no_grad():
        up = model(xe, torch.full_like(xe, T), torch.full_like(xe, c)).cpu().numpy().ravel()
    ue = exact(x, T, c)
    p = plt.plot(x, ue, lw=2.5, alpha=0.5)
    plt.plot(x, up, '--', color=p[0].get_color(), label=f'c={c}')
plt.plot(x, u0(x), 'k:', label='initial (t=0)')
plt.xlabel('x'); plt.ylabel('u @ t=T'); plt.grid(alpha=.3)
plt.title('Parametric surrogate: solid=exact, dashed=PINN, one model for all c')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Cell 5 -- Accuracy across the whole parameter range
cc = np.linspace(C_MIN, C_MAX, 16)
errs = []
for c in cc:
    with torch.no_grad():
        up = model(xe, torch.full_like(xe, T), torch.full_like(xe, float(c))).cpu().numpy().ravel()
    ue = exact(x, T, c)
    errs.append(np.sqrt(np.mean((up - ue) ** 2)))
plt.figure(figsize=(8, 3.5))
plt.plot(cc, errs, 'o-'); plt.xlabel('c'); plt.ylabel('L2 error @ t=T')
plt.title('One trained network stays accurate across all c'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'Mean L2 error over c in [{C_MIN},{C_MAX}]: {np.mean(errs):.3e}')

## Takeaways
- **One** training run produced a solver for the entire continuum of speeds
  $c\in[0.5,2.0]$. Each new $c$ is a millisecond forward pass, not a fresh simulation.
- The recipe generalizes: put **any** PDE parameter (diffusion, a boundary value, a
  geometry knob, a source amplitude) on the input layer and sample it during training
  to obtain a surrogate over that parameter.
- This is the engine behind real-time design sweeps, control, and uncertainty
  quantification — repeatedly asking "what if the parameter were different?" without
  paying for a full solve each time.

**Try it:** widen `C_MIN, C_MAX`, or add a second parameter (e.g. the Gaussian width)
as a 4th input and watch the same idea scale.